# A5 — Sequence Modeling with RNNs

**Goal:** Apply basic and advanced **RNN / LSTM / GRU, Encoder-Decoder** architectures to a sequence-based sentiment classification task.

---

## Dataset (Required)

### IMDB Movie Reviews (Binary Sentiment Classification)
**Access method (required):**
```python
from tensorflow.keras.datasets import imdb
(x_train, y_train), (x_test, y_test) = ...
```

You must create a validation split from training data and pad/truncate sequences to a fixed length.

---
## What you will implement

You will implement and compare the following **sequence models**:

- **Vanilla RNN**
- **LSTM**
- **GRU**
- **Stacked LSTM + Dropout**
- **Encoder–Decoder LSTM**
- **LSTM + GRU Hybrid**

---

## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [1]:
# ============================================================
# Q0) Environment Setup
# ============================================================

import os
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


TensorFlow: 2.19.0


---
## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q9)** and continues with the **Markdown responses (Q10–Q13)**.  
These correspond to the questions listed in the assignment description on **Canvas**. Follow the instructions provided in the **preceding Markdown cells** for each step.

### Tasks

This assignment focuses on **sequence modeling for text classification** using recurrent neural networks.

You will:

- Train and evaluate the following **sequence models**:
  - **Vanilla RNN**
  - **LSTM**
  - **GRU**

- Implement additional **advanced architectures**:
  - **Stacked LSTM + Dropout**
  - **Encoder–Decoder LSTM**
  - **LSTM + GRU Hybrid**

- Use the **IMDB Movie Reviews dataset** for **binary sentiment classification**.

- Perform a **comparative analysis** of the models, including:
  - training convergence behavior
  - validation and test performance
  - explanation of architectural differences across models

Ensure that all models are **computationally feasible** to train on **CPU-only environments** by using the recommended hyperparameters unless you have access to a GPU (e.g. Google Colab).

---

## Q1 — Load Dataset & Inspect

Use the **IMDB Movie Reviews dataset** from Keras and inspect its basic structure.

### Student Tasks

- Load the IMDB dataset using `tensorflow.keras.datasets.imdb` with a vocabulary size of **10,000 words** by frequency.

- Split the training data into **training** and **validation** sets.

- Inspect the dataset by:
  - Printing the **number of training and test samples**
  - Displaying the **label distribution (train)**
  - Printing one example **Sequence length (train)**

---

In [3]:
# ============================================================
# Question Q1 — Load IMDB Dataset (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Load the IMDB Movie Reviews dataset
# 2) Print the number of training and test examples
# 3) Check the label distribution in the training set
# 4) Inspect sequence length statistics
# ============================================================

import numpy as np
from tensorflow.keras.datasets import imdb

# TODO 1: Define vocabulary size
VOCAB_SIZE = 10000

# TODO 2: Load the IMDB dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# TODO 3: Print dataset sizes
print("Train examples:", len(x_train))
print("Test examples: ", len(x_test))

# TODO 4: Print label distribution
print("Label distribution (train):", np.bincount(y_train))

# TODO 5: Compute sequence length statistics
train_lengths = np.array([len(s) for s in x_train])

print(
    "Sequence length (train): min/median/max =",
    train_lengths.min(),
    int(np.median(train_lengths)),
    train_lengths.max()
)

Train examples: 25000
Test examples:  25000
Label distribution (train): [12500 12500]
Sequence length (train): min/median/max = 11 178 2494


---

## Q2 — Validation Split & Sequence Padding

Prepare the dataset for training by creating a **validation split** and converting all sequences to a **fixed length**.

### Student Tasks

- Create a **validation set** from the training data (`VAL_SIZE = 5000`).  
  Use a **deterministic split from the end of the training set**.

- Define a maximum sequence length **`MAX_LEN`** (e.g., 200–300 tokens) based on the sequence statistics observed in **Q1**.

- Apply **sequence padding and truncation** using `pad_sequences` so that all reviews have the same length:
  - Use **post-padding**
  - Use **post-truncation**

- Generate padded datasets for:
  - `x_train_pad`
  - `x_val_pad`
  - `x_test_pad`

- Keep it **consistent across all models**.

- Print the shapes of the padded arrays to confirm the preprocessing step completed successfully.

---

In [4]:
# ============================================================
# Question Q2 — Validation Split & Sequence Padding (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define MAX_LEN and validation size
# 2) Create a validation split from the training data
# 3) Pad/truncate sequences to a fixed length
# 4) Verify resulting dataset shapes
# ============================================================

from tensorflow.keras.preprocessing.sequence import pad_sequences

# TODO 1: Define sequence length and validation size
MAX_LEN = 200
VAL_SIZE = 5000

# TODO 2: Create validation split from the end of the training set
x_val, y_val = x_train[-VAL_SIZE:], y_train[-VAL_SIZE:]
x_train2, y_train2 = x_train[:-VAL_SIZE], y_train[:-VAL_SIZE]

# TODO 3: Apply padding and truncation
x_train_pad = pad_sequences(
    x_train2,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_val_pad = pad_sequences(
    x_val,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

# Print dataset shapes
print("x_train_pad:", x_train_pad.shape)
print("x_val_pad:  ", x_val_pad.shape)
print("x_test_pad: ", x_test_pad.shape)

x_train_pad: (20000, 200)
x_val_pad:   (5000, 200)
x_test_pad:  (25000, 200)


---

## Q3 — Build `tf.data` Pipelines

Create efficient **data pipelines** for training, validation, and testing using **TensorFlow `tf.data`**.

### Student Tasks

- Convert the padded datasets into **TensorFlow datasets** using `tf.data.Dataset.from_tensor_slices`.

- Create datasets for:
  - **training**
  - **validation**
  - **testing**

- Apply the following pipeline steps:
  - **shuffle** the training dataset
  - **batch** the datasets using an appropriate batch size
  - use **prefetching** to improve training performance

- Ensure the pipelines are ready to be used directly in **model training with `model.fit()`**.


---

In [5]:
# ============================================================
# Question Q3 — tf.data Pipelines (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define batch size and AUTOTUNE
# 2) Create TensorFlow datasets from the padded arrays
# 3) Apply shuffle, batch, and prefetch operations
# 4) Prepare pipelines for training, validation, and testing
# ============================================================

# TODO 1: Define batch size and autotune
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# TODO 2: Create training dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train_pad, y_train2))

# TODO 3: Apply shuffle, batch, and prefetch
train_ds = train_ds.shuffle(len(x_train_pad), seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 4: Create validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((x_val_pad, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 5: Create test dataset
test_ds = tf.data.Dataset.from_tensor_slices((x_test_pad, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Pipelines ready.")

Pipelines ready.


---

## Q4 — Define Training Utilities

In this step, you will create **reusable helper functions** that simplify model training, evaluation, and experiment management.

These utilities will be used for **all models later in the assignment**.

### Student Tasks

1. **Create training callbacks**

   Define a function that returns commonly used training callbacks, including:

   - **EarlyStopping** to stop training when validation performance stops improving.
   - **ReduceLROnPlateau** to automatically reduce the learning rate when validation loss plateaus.
   - **ModelCheckpoint** to save the **best-performing model** during training.

2. **Define a model compilation function**

   Implement a function that compiles a model using:

   - **Adam optimizer**
   - **Binary cross-entropy loss** for sentiment classification
   - **Accuracy** as the evaluation metric

3. **Create a training function**

   Implement a function that trains a model using:

   - the **training dataset**
   - the **validation dataset**
   - the callbacks defined above

4. **Create an evaluation function**

   Implement a function that evaluates a trained model on a dataset and reports:

   - **loss**
   - **accuracy**

These functions will help keep the notebook **organized, reusable, and consistent across experiments**.

---

In [7]:
# ============================================================
# Question Q4 — Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define callbacks for training
# 2) Compile the model with optimizer, loss, and metrics
# 3) Train the model using training and validation datasets
# 4) Evaluate the trained model on a dataset
# ============================================================

# TODO 1: Define training callbacks
def build_callbacks(run_name: str):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"{run_name}.keras",
            monitor="val_accuracy",
            save_best_only=True,
        ),
    ]


# TODO 2: Compile model
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=["accuracy"],
    )
    return model


# TODO 3: Train model
def train_model(model, run_name: str, epochs=8):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=build_callbacks(run_name),
        verbose=1,
    )
    return history


# TODO 4: Evaluate model
def evaluate_model(model, name: str, ds):
    loss, acc = model.evaluate(ds, verbose=0)
    print(f"{name}: loss={loss:.4f}, acc={acc:.4f}")
    return loss, acc


---

## Q5 — Model A: Vanilla RNN

Build a **Vanilla RNN** model for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add a **SimpleRNN layer** for sequence processing.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.

---

In [9]:
# ============================================================
# Question Q5 — Model A: Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define embedding dimension and RNN units
# 2) Build a Sequential RNN model
# 3) Add Input, Embedding, RNN, and Dense layers
# 4) Compile the model
# 5) Display the model summary
# ============================================================

from tensorflow.keras import layers

# TODO 1: Define model hyperparameters (e.g., 128, 128)
EMBED_DIM = 128
RNN_UNITS = 128

# TODO 2: Build the Sequential model
rnn_model = tf.keras.Sequential([

    # TODO 3: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 4: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 5: Simple RNN layer
    layers.SimpleRNN(RNN_UNITS),

    # TODO 6: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="vanilla_rnn")

# TODO 7: Compile the model
rnn_model = compile_model(rnn_model, lr=1e-3)

# Print model summary
rnn_model.summary()

Model: "vanilla_rnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# ============================================================
# Train and Evaluate Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the Vanilla RNN model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the RNN model
history_rnn = train_model(rnn_model, run_name="proj5_vanilla_rnn", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(rnn_model, "Validation (RNN)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(rnn_model, "Test (RNN)", test_ds)


Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 44s 140ms/step - accuracy: 0.7322 - loss: 0.4957 - val_accuracy: 0.5614 - val_loss: 0.7521 - learning_rate: 3.1250e-05
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 46s 146ms/step - accuracy: 0.7470 - loss: 0.4705 - val_accuracy: 0.5800 - val_loss: 0.7697 - learning_rate: 3.1250e-05
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 42s 135ms/step - accuracy: 0.8018 - loss: 0.4227 - val_accuracy: 0.6704 - val_loss: 0.7016 - learning_rate: 1.5625e-05
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 45s 144ms/step - accuracy: 0.8360 - loss: 0.3890 - val_accuracy: 0.6652 - val_loss: 0.7195 - learning_rate: 1.5625e-05
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 128ms/step - accuracy: 0.8200 - loss: 0.4075 - val_accuracy: 0.6712 - val_loss: 0.7015 - learning_rate: 7.8125e-06
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 155ms/step - accuracy: 0.8335 - loss: 0.3871 - val_accuracy: 0.6798 - val_loss: 0.6893 - learning_rate: 3.9063e-06
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 75s 133ms/ste

(0.6942145228385925, 0.682479977607727)

---

## Q6 — Model B: LSTM

Build an **LSTM-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add an **LSTM layer** to capture long-term dependencies in sequences.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [13]:
# ============================================================
# Question Q6 — Model B: LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build an LSTM-based sequence model
# 2) Add Input, Embedding, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential LSTM model
lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm")

# TODO 6: Compile the model
lstm_model = compile_model(lstm_model, lr=1e-3)

# Print model summary
lstm_model.summary()

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,411,713 (5.39 MB)

 Trainable params: 1,411,713 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# ============================================================
# Train and Evaluate LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the LSTM model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the LSTM model
history_lstm = train_model(lstm_model, run_name="proj5_lstm", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(lstm_model, "Validation (LSTM)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(lstm_model, "Test (LSTM)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 167s 526ms/step - accuracy: 0.5267 - loss: 0.6903 - val_accuracy: 0.5376 - val_loss: 0.6872 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 161s 514ms/step - accuracy: 0.5919 - loss: 0.6494 - val_accuracy: 0.6270 - val_loss: 0.6351 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 154s 491ms/step - accuracy: 0.6529 - loss: 0.6003 - val_accuracy: 0.6094 - val_loss: 0.7200 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 144s 460ms/step - accuracy: 0.8243 - loss: 0.4198 - val_accuracy: 0.7910 - val_loss: 0.4843 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 143s 457ms/step - accuracy: 0.8543 - loss: 0.3634 - val_accuracy: 0.8038 - val_loss: 0.4695 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 150s 480ms/step - accuracy: 0.8831 - loss: 0.3044 - val_accuracy: 0.8140 - val_loss: 0.4426 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 143s 457ms/step - a

(0.46794331073760986, 0.8095200061798096)

---

## Q7 — Model C: GRU

Build a **GRU-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add **Embedding → GRU → Dense(sigmoid)** layers.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [15]:
# ============================================================
# Question Q7 — Model C: GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a GRU-based sequence model
# 2) Add Input, Embedding, GRU, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential GRU model
gru_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: GRU layer
    layers.GRU(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="gru")

# TODO 6: Compile the model
gru_model = compile_model(gru_model, lr=1e-3)

# Print model summary
gru_model.summary()

Model: "gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,379,201 (5.26 MB)

 Trainable params: 1,379,201 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# ============================================================
# Train and Evaluate GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the GRU model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the GRU model
history_gru = train_model(gru_model, run_name="proj5_gru", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(gru_model, "Validation (GRU)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(gru_model, "Test (GRU)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 144s 450ms/step - accuracy: 0.5189 - loss: 0.6917 - val_accuracy: 0.5628 - val_loss: 0.6765 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 146s 468ms/step - accuracy: 0.6259 - loss: 0.6406 - val_accuracy: 0.6716 - val_loss: 0.5925 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 453ms/step - accuracy: 0.8091 - loss: 0.4295 - val_accuracy: 0.8194 - val_loss: 0.4361 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 452ms/step - accuracy: 0.8462 - loss: 0.3501 - val_accuracy: 0.8508 - val_loss: 0.3642 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 143s 456ms/step - accuracy: 0.9233 - loss: 0.1994 - val_accuracy: 0.8568 - val_loss: 0.3628 - learning_rate: 0.0010
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 454ms/step - accuracy: 0.9600 - loss: 0.1194 - val_accuracy: 0.8568 - val_loss: 0.4103 - learning_rate: 0.0010
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 454ms/step - accuracy: 0.9

(0.5655039548873901, 0.8397200107574463)

---

## 8) Complex Model D — Stacked LSTM with Dropout

In this question, build a **deeper LSTM-based sequence classifier** using two recurrent layers.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `Dropout(...)`
  - `LSTM(...)`
  - `Dense(1, activation="sigmoid")`
- Train the model using the same optimizer and callbacks.
- Evaluate on validation and test sets.
- Compare it with the single-layer LSTM from Q6.

### Goal
Study whether **stacking recurrent layers** helps the model learn richer sequential sentiment patterns.


---

In [17]:
# ============================================================
# Question Q8 — Complex Model D: Stacked LSTM + Dropout (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a stacked LSTM model with dropout
# 2) Add Input, Embedding, LSTM, Dropout, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential stacked LSTM model
stacked_lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: First LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: Dropout layer
    layers.Dropout(0.3),

    # TODO 6: Second LSTM layer
    layers.LSTM(RNN_UNITS // 2),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="stacked_lstm_dropout")

# TODO 8: Compile the model
stacked_lstm_model = compile_model(stacked_lstm_model, lr=1e-3)

# Print model summary
stacked_lstm_model.summary()

# TODO 9: Train the model
history_stacked_lstm = train_model(
    stacked_lstm_model,
    run_name="proj5_stacked_lstm_dropout",
    epochs=8
)

# TODO 10: Evaluate on validation set
evaluate_model(stacked_lstm_model, "Validation (Stacked LSTM + Dropout)", val_ds)

# TODO 11: Evaluate on test set
evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)


Model: "stacked_lstm_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 200, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 200, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,057 (5.57 MB)

 Trainable params: 1,461,057 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 218s 686ms/step - accuracy: 0.5282 - loss: 0.6896 - val_accuracy: 0.6082 - val_loss: 0.6441 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 212s 677ms/step - accuracy: 0.6249 - loss: 0.6345 - val_accuracy: 0.7002 - val_loss: 0.5882 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 213s 682ms/step - accuracy: 0.6245 - loss: 0.6359 - val_accuracy: 0.5754 - val_loss: 0.6671 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 212s 678ms/step - accuracy: 0.6189 - loss: 0.6367 - val_accuracy: 0.7378 - val_loss: 0.6113 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 216s 692ms/step - accuracy: 0.8191 - loss: 0.4548 - val_accuracy: 0.8198 - val_loss: 0.4336 - learning_rate: 2.5000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 211s 674ms/step - accuracy: 0.8626 - loss: 0.3390 - val_accuracy: 0.8268 - val_loss: 0.3875 - learning_rate: 2.5000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 213s 681ms/step - a

(0.4360378086566925, 0.8261200189590454)

---

## 9) Complex Model E — Encoder–Decoder LSTM Classifier

In this question, build an **encoder–decoder style recurrent model** for sentiment classification.

### Idea
- The **encoder LSTM** reads the review and produces a compact context representation.
- A `RepeatVector` creates a short decoded sequence from that context.
- A **decoder LSTM** transforms the context into a richer hidden representation.
- A final dense layer predicts the review sentiment.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM` encoder
  - `RepeatVector(1)`
  - `LSTM` decoder
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it with the simpler one-layer LSTM.

### Goal
Explore whether a **more structured encoder–decoder design** is useful for sequence classification, even though it is more common in seq2seq tasks.


---

In [19]:
# ============================================================
# Question Q9 — Complex Model E: Encoder-Decoder LSTM Classifier (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define the input layer
# 2) Add Embedding, Encoder LSTM, RepeatVector, Decoder LSTM, and Dense layers
# 3) Build the functional model
# 4) Compile the model using the training utility
# 5) Train the model
# 6) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Define input layer
encdec_inputs = layers.Input(shape=(MAX_LEN,))

# TODO 2: Embedding layer
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(encdec_inputs)

# TODO 3: Encoder LSTM
encoded = layers.LSTM(RNN_UNITS)(x)

# TODO 4: Repeat encoded representation
decoded = layers.RepeatVector(1)(encoded)

# TODO 5: Decoder LSTM
decoded = layers.LSTM(RNN_UNITS // 2)(decoded)

# TODO 6: Output layer
encdec_outputs = layers.Dense(1, activation="sigmoid")(decoded)

# TODO 7: Build functional model
encdec_model = tf.keras.Model(encdec_inputs, encdec_outputs, name="encdec_lstm_classifier")

# TODO 8: Compile the model
encdec_model = compile_model(encdec_model, lr=1e-3)

# TODO 9: Print model summary
encdec_model.summary()

# TODO 10: Train the model
history_encdec = train_model(
    encdec_model,
    run_name="proj5_encdec_lstm",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(encdec_model, "Validation (Encoder-Decoder LSTM)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)


Model: "encdec_lstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,057 (5.57 MB)

 Trainable params: 1,461,057 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 148s 460ms/step - accuracy: 0.5198 - loss: 0.6905 - val_accuracy: 0.5218 - val_loss: 0.6927 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 202s 460ms/step - accuracy: 0.5865 - loss: 0.6684 - val_accuracy: 0.5308 - val_loss: 0.6965 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 144s 460ms/step - accuracy: 0.6485 - loss: 0.6208 - val_accuracy: 0.6878 - val_loss: 0.5357 - learning_rate: 5.0000e-04
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 145s 464ms/step - accuracy: 0.8283 - loss: 0.3902 - val_accuracy: 0.8444 - val_loss: 0.3611 - learning_rate: 5.0000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 144s 459ms/step - accuracy: 0.8914 - loss: 0.2742 - val_accuracy: 0.8632 - val_loss: 0.3337 - learning_rate: 5.0000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 203s 464ms/step - accuracy: 0.9186 - loss: 0.2122 - val_accuracy: 0.8712 - val_loss: 0.3220 - learning_rate: 5.0000e-04
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 144s 460ms/step

(0.397901326417923, 0.8490399718284607)

---

## 10) Complex Model F — LSTM + GRU Hybrid

In this question, build a **hybrid recurrent architecture** that combines LSTM and GRU layers without using bidirectional processing.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `GRU(...)`
  - `Dropout`
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it against all earlier models in terms of accuracy and complexity.

### Goal
Test whether combining **LSTM-based memory** with a **GRU-based final sequence encoder** captures richer sentiment patterns than a single recurrent layer.


---

In [20]:
# ============================================================
# Question Q10 — Complex Model F: LSTM + GRU Hybrid (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a hybrid sequence model using LSTM and GRU layers
# 2) Add Input, Embedding, LSTM, GRU, Dropout, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential hybrid model
hybrid_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(MAX_LEN,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: GRU layer
    layers.GRU(RNN_UNITS // 2, dropout=0.2, recurrent_dropout=0.0),

    # TODO 6: Dropout layer
    layers.Dropout(0.3),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm_gru_hybrid")

# TODO 8: Compile the model
hybrid_model = compile_model(hybrid_model, lr=1e-3)

# TODO 9: Print model summary
hybrid_model.summary()

# TODO 10: Train the model
history_hybrid = train_model(
    hybrid_model,
    run_name="proj5_lstm_gru_hybrid",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(hybrid_model, "Validation (LSTM + GRU Hybrid)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)


Model: "lstm_gru_hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 200, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,448,897 (5.53 MB)

 Trainable params: 1,448,897 (5.53 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 222s 692ms/step - accuracy: 0.5134 - loss: 0.6986 - val_accuracy: 0.5476 - val_loss: 0.6880 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 217s 692ms/step - accuracy: 0.5409 - loss: 0.6889 - val_accuracy: 0.5530 - val_loss: 0.6797 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 216s 689ms/step - accuracy: 0.5525 - loss: 0.6782 - val_accuracy: 0.5744 - val_loss: 0.6667 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 219s 700ms/step - accuracy: 0.6690 - loss: 0.5859 - val_accuracy: 0.7980 - val_loss: 0.4724 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 262s 699ms/step - accuracy: 0.8306 - loss: 0.3915 - val_accuracy: 0.8462 - val_loss: 0.3647 - learning_rate: 0.0010
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 227s 726ms/step - accuracy: 0.9068 - loss: 0.2399 - val_accuracy: 0.8620 - val_loss: 0.3453 - learning_rate: 0.0010
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 213s 681ms/step - accuracy: 0.9

(0.4845578968524933, 0.8395199775695801)

---

## Q11 — Performance Comparison Table

Create a compact table comparing all completed models:

- Vanilla RNN
- LSTM
- GRU
- Stacked LSTM + Dropout
- Encoder–Decoder LSTM
- LSTM + GRU Hybrid

### Student Tasks
- Create a comparison table summarizing the results of all models.
- Include validation accuracy and test accuracy.
- Identify the best-performing recurrent model.
- Briefly comment on whether more complex architectures were helpful.


---

In [21]:
# ============================================================
# Question Q11 — Performance Comparison Table (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define helper functions for validation and training accuracy
# 2) Evaluate all trained models on the test dataset
# 3) Store model names and metrics in summary rows
# 4) Create a pandas DataFrame for comparison
# 5) Sort the table by test accuracy
# ============================================================

import pandas as pd

# TODO 1: Best validation accuracy from training history
def best_val_acc(history):
    return np.max(history.history.get("val_accuracy", [np.nan]))

# TODO 2: Final training accuracy from training history
def final_train_acc(history):
    return history.history.get("accuracy", [np.nan])[-1]

summary_rows = []

# TODO 3: Evaluate all models on the test set
_, rnn_test_acc = evaluate_model(rnn_model, "Test (Vanilla RNN)", test_ds)
_, lstm_test_acc = evaluate_model(lstm_model, "Test (LSTM)", test_ds)
_, gru_test_acc = evaluate_model(gru_model, "Test (GRU)", test_ds)
_, stacked_lstm_test_acc = evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)
_, encdec_test_acc = evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)
_, hybrid_test_acc = evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)

# TODO 4: Append summary rows
summary_rows.append(["Vanilla RNN", final_train_acc(history_rnn), best_val_acc(history_rnn), rnn_test_acc])
summary_rows.append(["LSTM", final_train_acc(history_lstm), best_val_acc(history_lstm), lstm_test_acc])
summary_rows.append(["GRU", final_train_acc(history_gru), best_val_acc(history_gru), gru_test_acc])
summary_rows.append(["Stacked LSTM + Dropout", final_train_acc(history_stacked_lstm), best_val_acc(history_stacked_lstm), stacked_lstm_test_acc])
summary_rows.append(["Encoder-Decoder LSTM", final_train_acc(history_encdec), best_val_acc(history_encdec), encdec_test_acc])
summary_rows.append(["LSTM + GRU Hybrid", final_train_acc(history_hybrid), best_val_acc(history_hybrid), hybrid_test_acc])

# TODO 5: Create DataFrame
results_df = pd.DataFrame(
    summary_rows,
    columns=["Model", "Final Train Acc", "Best Val Acc", "Test Acc"]
)

# TODO 6: Sort by test accuracy
results_df = results_df.sort_values(by="Test Acc", ascending=False).reset_index(drop=True)

# Display results
results_df


Test (Vanilla RNN): loss=0.6942, acc=0.6825
Test (LSTM): loss=0.4679, acc=0.8095
Test (GRU): loss=0.5655, acc=0.8397
Test (Stacked LSTM + Dropout): loss=0.4360, acc=0.8261
Test (Encoder-Decoder LSTM): loss=0.3979, acc=0.8490
Test (LSTM + GRU Hybrid): loss=0.4846, acc=0.8395


,Model,Final Train Acc,Best Val Acc,Test Acc
0,Encoder-Decoder LSTM,0.96040,0.8740,0.84904
1,GRU,0.99385,0.8576,0.83972
2,LSTM + GRU Hybrid,0.96600,0.8642,0.83952
3,Stacked LSTM + Dropout,0.90485,0.8516,0.82612
4,LSTM,0.91530,0.8298,0.80952
5,Vanilla RNN,0.84350,0.6818,0.68248


---

# Results and Discussions

Use the results above to answer the discussion questions below. You may revise the answers based on your actual experimental results (**Q1-Q11**).

---

## **Q12** — Which model performed best overall, and why might it outperform the others?  

**Answer:** ...

The model that maintains good validation performance and steady training along with the highest test accuracy is the best model. This is typically the GRU, LSTM, or LSTM + GRU hybrid.

Because they employ gates to better regulate how data is stored and transmitted throughout the network, these models perform better than the vanilla RNN. This helps them avoid problems like vanishing gradients and capture long-term dependencies in text.

When GRU works best, it's usually because it strikes a nice balance between efficiency, accuracy, and stability. Better memory handling results in better emotion categorization, regardless of whether LSTM or the hybrid prevails.

## **Q13** — Why does a vanilla RNN usually struggle more on long reviews?  

**Answer:** ...

A vanilla RNN usually struggles on long reviews because it must continuously send data through numerous time steps using the same recurrent transformation. Gradients can get extremely small or very large during backpropagation via time. The **vanishing-gradient** or **exploding-gradient** problem results from this.

Important words that determine polarity may appear far apart in a review for sentiment classification. A vanilla RNN struggles to capture long-range dependencies because it frequently forgets earlier relevant context before reaching the end of the sequence. This is addressed by LSTM and GRU models, which are far more dependable on lengthy text sequences because they use gating mechanisms that determine what to retain, update, or forget.

## **Q14** — Why might a more complex model not always outperform a simpler GRU or LSTM?  

**Answer:** ...

A more complex model doesn’t always perform better than a GRU or LSTM because more complexity doesn’t guarantee better results.

Deeper or hybrid models are harder to train, can overfit the data, and may add unnecessary complexity for a task like sentiment classification. Also, simpler models like GRU or LSTM often train faster and perform well, especially under limited resources.

So, the best model is the one that balances accuracy, stability, and efficiency—not necessarily the most complex one.

## **Q15** — What is the main idea behind the encoder–decoder classifier used here?  

**Answer:** ...


The main idea behind the encoder–decoder classifier is to first **encode** the full input review into a compact context representation and then **decode** that representation into features useful for final classification.

The padded review sequence is read by the encoder LSTM in this notebook, which then compresses it into a hidden form. From that representation, a RepeatVector generates a brief sequence, which is then processed by the decoder LSTM before the final sigmoid layer generates the sentiment forecast.

Sequence-to-sequence learning served as the model for this architecture. The encoder-decoder design encourages the model to condense the entire review into a dense representation before making the positive/negative judgment, even if the job at hand is classification rather than sequence synthesis. Its strength is structured sequence abstraction, but because it adds more complexity, it does not necessarily perform better for binary sentiment classification than a more straightforward GRU or LSTM.


### 🎉 Congratulations!

You have successfully completed **A5 — Sequence Modeling with RNNs**. Excellent work applying and comparing **RNN, LSTM, and GRU architectures** for a **sequence-based text classification** task using the **IMDB Movie Reviews dataset**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q11)** have been executed successfully
- All **Markdown responses (Q12–Q15)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.